# 02 - Limpieza inicial y consolidación

**Objetivo.** Normalizar categorías, fechas, textos, barrios y faltantes encubiertos; crear flags de nulidad por columna; revisar patrones de faltantes; y generar datasets procesados trazables en `data/processed`.

Este notebook prepara la base para el EDA, pero evita extraer conclusiones definitivas sobre hipótesis. La pregunta central acá es: **qué datos son comparables, qué datos requieren cautela y qué decisiones de limpieza quedan documentadas**.

## Convenciones de nombres

- `df_raw_*`: datos crudos leídos desde `data/raw`.
- `df_preprocesado_*`: versión limpiada inicial por fuente/operación.
- `df_publicaciones_consolidadas`: base integrada con columnas canónicas.
- `*_is_null`: flag por fila que vale `1` si la variable original está faltante luego de reemplazar valores inválidos por nulo.

In [ ]:
from pathlib import Path
import json
import re
import unicodedata
from datetime import datetime

import numpy as np
import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 80)
pd.set_option('display.width', 160)


def find_repo_root(start=None):
    """Busca la raíz del repo desde la ubicación actual del notebook.

    Funciona aunque el notebook se ejecute desde /notebooks, /data/processed
    o desde la raíz del proyecto.
    """
    start = Path(start or Path.cwd()).resolve()
    candidates = [start, *start.parents]
    for path in candidates:
        if (path / 'data' / 'raw').exists() and (path / 'README.md').exists():
            return path
    raise FileNotFoundError(
        'No se encontró la raíz del repo. Ejecutar el notebook dentro del proyecto '
        'TP_analitica_descriptiva_grupo1_2q2026.'
    )

REPO_ROOT = find_repo_root()
RAW_DIR = REPO_ROOT / 'data' / 'raw'
PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
REPORTS_DIR = PROCESSED_DIR / 'reports'
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print('REPO_ROOT:', REPO_ROOT)
print('RAW_DIR:', RAW_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)

## Carga de crudos

Se replica la carga del notebook 01 para que este notebook sea ejecutable de forma independiente.

In [ ]:
INVALID_MARKERS = {
    '', ' ', 'nan', 'NaN', 'NAN', 'none', 'None', 'NONE', 'null', 'NULL',
    's/d', 'S/D', 'sd', 'SD', 'sin dato', 'Sin dato', 'Sin Datos', 'sin datos',
    'no informa', 'No informa', 'no informado', 'No informado', 'n/a', 'N/A',
    '-', '--', '---', '?', '99999', '-99999', '-9999', '9999'
}

RAW_SOURCES = {
    'df_raw_meli_alq': {
        'descripcion': 'Mercado Libre - alquiler permanente',
        'paths': sorted((RAW_DIR / 'mercadolibre_inmuebles_alquiler').glob('meli_alq_*.csv')),
    },
    'df_raw_meli_alq_temp': {
        'descripcion': 'Mercado Libre - alquiler temporario',
        'paths': sorted((RAW_DIR / 'mercadolibre_inmuebles_alquiler temporario').glob('meli_alq_temp_*.csv')),
    },
    'df_raw_meli_vtas': {
        'descripcion': 'Mercado Libre - venta',
        'paths': sorted((RAW_DIR / 'mercadolibre_inmuebles_ventas').glob('meli_vtas_*.csv')),
    },
    'df_raw_argenprop': {
        'descripcion': 'Argenprop - publicaciones extraídas',
        'paths': [RAW_DIR / 'argenprop' / 'propiedades_argenprop.csv'],
    },
    'df_raw_zonaprop': {
        'descripcion': 'Zonaprop - publicaciones extraídas',
        'paths': [RAW_DIR / 'zonaprop' / 'propiedades_zonaprop.csv'],
    },
    'df_raw_airbnb_norm': {
        'descripcion': 'Airbnb CABA normalizado mensual. Base prioritaria para análisis comparable',
        'paths': [RAW_DIR / 'airbnb' / 'airbnb_caba_normalizado_mensual.csv'],
    },
}


def read_csv_safely(path):
    return pd.read_csv(path, low_memory=False)


def load_many(paths, source_name):
    existing = [Path(p) for p in paths if Path(p).exists()]
    if not existing:
        print(f'ATENCIÓN: no se encontraron archivos para {source_name}')
        return pd.DataFrame()
    frames = []
    for path in existing:
        df = read_csv_safely(path)
        df['_archivo_origen'] = str(path.relative_to(REPO_ROOT))
        frames.append(df)
    return pd.concat(frames, ignore_index=True, sort=False)

raw_dfs = {}
for df_name, spec in RAW_SOURCES.items():
    raw_dfs[df_name] = load_many(spec['paths'], df_name)
    globals()[df_name] = raw_dfs[df_name]
    print(f"{df_name}: {raw_dfs[df_name].shape} | {spec['descripcion']}")

## Funciones de limpieza y normalización

Incluyen:

- reemplazo de valores inválidos que indican faltantes;
- normalización de mayúsculas/minúsculas, espacios y tildes;
- normalización inicial de barrios;
- parseo de fechas;
- creación de variables canónicas comparables;
- flags de nulidad.

In [ ]:
INVALID_MARKERS = {
    '', ' ', 'nan', 'NaN', 'NAN', 'none', 'None', 'NONE', 'null', 'NULL',
    's/d', 'S/D', 'sd', 'SD', 'sin dato', 'Sin dato', 'Sin Datos', 'sin datos',
    'no informa', 'No informa', 'no informado', 'No informado', 'n/a', 'N/A',
    '-', '--', '---', '?', '99999', '-99999', '-9999', '9999'
}

CABA_BARRIOS = {
    'agronomia':'agronomia', 'almagro':'almagro', 'balvanera':'balvanera', 'barracas':'barracas',
    'belgrano':'belgrano', 'boedo':'boedo', 'caballito':'caballito', 'chacarita':'chacarita',
    'coghlan':'coghlan', 'colegiales':'colegiales', 'constitucion':'constitucion', 'flores':'flores',
    'floresta':'floresta', 'la boca':'la boca', 'liniers':'liniers', 'mataderos':'mataderos',
    'monserrat':'monserrat', 'monte castro':'monte castro', 'nueva pompeya':'nueva pompeya',
    'nunez':'nunez', 'palermo':'palermo', 'parque avellaneda':'parque avellaneda',
    'parque chacabuco':'parque chacabuco', 'parque chas':'parque chas',
    'parque patricios':'parque patricios', 'paternal':'paternal', 'puerto madero':'puerto madero',
    'recoleta':'recoleta', 'retiro':'retiro', 'saavedra':'saavedra', 'san cristobal':'san cristobal',
    'san nicolas':'san nicolas', 'san telmo':'san telmo', 'velez sarsfield':'velez sarsfield',
    'versalles':'versalles', 'villa crespo':'villa crespo', 'villa del parque':'villa del parque',
    'villa devoto':'villa devoto', 'villa gral mitre':'villa general mitre',
    'villa general mitre':'villa general mitre', 'villa lugano':'villa lugano',
    'villa luro':'villa luro', 'villa ortuzar':'villa ortuzar', 'villa pueyrredon':'villa pueyrredon',
    'villa real':'villa real', 'villa riachuelo':'villa riachuelo', 'villa santa rita':'villa santa rita',
    'villa soldati':'villa soldati', 'villa urquiza':'villa urquiza'
}

BARRIO_ALIASES = {
    'nuñez': 'nunez', 'nunez': 'nunez', 'agronomía': 'agronomia',
    'san nicolás': 'san nicolas', 'san cristóbal': 'san cristobal',
    'vélez sársfield': 'velez sarsfield', 'villa general mitre': 'villa general mitre',
    'villa gral. mitre': 'villa general mitre', 'villa gral mitre': 'villa general mitre',
    'palermo chico': 'palermo', 'palermo hollywood': 'palermo', 'palermo soho': 'palermo',
    'las cañitas': 'palermo', 'belgrano chico': 'belgrano', 'barrio norte': 'recoleta',
    'capital federal': np.nan, 'ciudad autonoma de buenos aires': np.nan,
    'ciudad autónoma de buenos aires': np.nan, 'caba': np.nan,
}


def strip_accents(value):
    if pd.isna(value):
        return np.nan
    text = str(value)
    text = unicodedata.normalize('NFKD', text)
    return ''.join(ch for ch in text if not unicodedata.combining(ch))


def clean_text(value, lower=True):
    if pd.isna(value):
        return np.nan
    text = str(value).replace('\xa0', ' ').strip()
    text = re.sub(r'\s+', ' ', text)
    if text in INVALID_MARKERS:
        return np.nan
    if lower:
        text = strip_accents(text).lower()
    return text


def normalize_barrio(value):
    text = clean_text(value, lower=True)
    if pd.isna(text):
        return np.nan
    text = BARRIO_ALIASES.get(text, text)
    if pd.isna(text):
        return np.nan
    if text in CABA_BARRIOS:
        return CABA_BARRIOS[text]
    return text


def normalize_operation(value):
    text = clean_text(value, lower=True)
    if pd.isna(text):
        return np.nan
    if 'tempor' in text:
        return 'alquiler_temporal'
    if 'alquiler' in text:
        return 'alquiler'
    if 'venta' in text:
        return 'venta'
    return text.replace(' ', '_')


def normalize_currency(value):
    text = clean_text(value, lower=False)
    if pd.isna(text):
        return np.nan
    t = strip_accents(str(text)).upper().replace('$', 'ARS').strip()
    if t in {'US$', 'U$S', 'USD'}:
        return 'USD'
    if t in {'ARS', 'PESOS', 'PESO'}:
        return 'ARS'
    return t


def replace_invalid_markers(df):
    out = df.copy()
    obj_cols = out.select_dtypes(include=['object', 'string']).columns
    for col in obj_cols:
        stripped = out[col].astype('string').str.strip()
        out.loc[stripped.isin(INVALID_MARKERS), col] = np.nan
    return out


def parse_date_columns(df):
    out = df.copy()
    date_like = [c for c in out.columns if any(token in c.lower() for token in ['fecha', 'date', 'scraped_at'])]
    for col in date_like:
        out[col] = pd.to_datetime(out[col], errors='coerce', utc=True)
    return out


def add_null_flags(df, exclude=None):
    out = df.copy()
    exclude = set(exclude or [])
    original_cols = [c for c in out.columns if c not in exclude and not c.endswith('_is_null')]
    for col in original_cols:
        if out[col].isna().any():
            out[f'{col}_is_null'] = out[col].isna().astype('int8')
    return out


def to_numeric(df, cols):
    out = df.copy()
    for col in cols:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors='coerce')
    return out


def first_existing(df, columns, default=np.nan):
    existing = [c for c in columns if c in df.columns]
    if not existing:
        return pd.Series(default, index=df.index)
    result = df[existing[0]].copy()
    for col in existing[1:]:
        result = result.combine_first(df[col])
    return result


def canonical_empty(df):
    return pd.DataFrame(index=df.index)


def standardize_meli(df, operacion_forzada=None):
    out = replace_invalid_markers(df)
    out = parse_date_columns(out)
    out = to_numeric(out, [
        'precio', 'superficie_min_m2', 'superficie_max_m2', 'ambientes_min', 'ambientes_max',
        'banios_min', 'banios_max', 'dormitorios_min', 'dormitorios_max', 'comuna',
        'precio_m2_min', 'precio_m2_max', 'altura'
    ])
    out['fuente_norm'] = 'mercado_libre'
    out['tipo_operacion_norm'] = operacion_forzada or out.get('tipo_operacion', pd.Series(index=out.index, dtype='object')).map(normalize_operation)
    out['barrio_norm'] = out.get('barrio', pd.Series(index=out.index, dtype='object')).map(normalize_barrio)
    out['moneda_norm'] = out.get('moneda', pd.Series(index=out.index, dtype='object')).map(normalize_currency)
    out['precio_m2_ref'] = first_existing(out, ['precio_m2_min'])
    mask_precio_m2 = out['precio_m2_ref'].isna() & out.get('precio', np.nan).notna() & out.get('superficie_min_m2', np.nan).gt(0)
    out.loc[mask_precio_m2, 'precio_m2_ref'] = out.loc[mask_precio_m2, 'precio'] / out.loc[mask_precio_m2, 'superficie_min_m2']
    out['id_publicacion'] = out.get('item_id')
    out['superficie_ref_m2'] = first_existing(out, ['superficie_min_m2', 'superficie_max_m2'])
    out['ambientes_ref'] = first_existing(out, ['ambientes_min', 'ambientes_max'])
    out['banios_ref'] = first_existing(out, ['banios_min', 'banios_max'])
    out['dormitorios_ref'] = first_existing(out, ['dormitorios_min', 'dormitorios_max'])
    out['fecha_extraccion'] = out.get('scraped_at_utc')
    return out


def standardize_portal(df, fuente_norm):
    out = replace_invalid_markers(df)
    out = parse_date_columns(out)
    out = to_numeric(out, [
        'precio', 'superficie_total_m2', 'superficie_cubierta_m2', 'ambientes', 'banios',
        'dormitorios', 'comuna', 'precio_m2_calculado', 'latitud', 'longitud', 'altura'
    ])
    out['fuente_norm'] = fuente_norm
    out['tipo_operacion_norm'] = out.get('tipo_operacion', pd.Series(index=out.index, dtype='object')).map(normalize_operation)
    out['barrio_norm'] = first_existing(out, ['barrio', 'localidad', 'calle']).map(normalize_barrio)
    out['moneda_norm'] = out.get('moneda', pd.Series(index=out.index, dtype='object')).map(normalize_currency)
    out['precio_m2_ref'] = first_existing(out, ['precio_m2_calculado'])
    mask_precio_m2 = out['precio_m2_ref'].isna() & out.get('precio', np.nan).notna() & out.get('superficie_total_m2', np.nan).gt(0)
    out.loc[mask_precio_m2, 'precio_m2_ref'] = out.loc[mask_precio_m2, 'precio'] / out.loc[mask_precio_m2, 'superficie_total_m2']
    out['id_publicacion'] = out.get('property_id')
    out['superficie_ref_m2'] = first_existing(out, ['superficie_total_m2', 'superficie_cubierta_m2'])
    out['ambientes_ref'] = out.get('ambientes')
    out['banios_ref'] = out.get('banios')
    out['dormitorios_ref'] = out.get('dormitorios')
    out['fecha_extraccion'] = out.get('scraped_at_utc')
    return out


def standardize_airbnb(df):
    out = replace_invalid_markers(df)
    out = parse_date_columns(out)
    out = to_numeric(out, [
        'precio', 'ambientes', 'banios', 'dormitorios', 'comuna', 'latitud', 'longitud',
        'rating', 'reviews', 'capacidad', 'camas'
    ])
    out['fuente_norm'] = 'airbnb'
    out['tipo_operacion_norm'] = 'alquiler_temporal'
    out['barrio_norm'] = first_existing(out, ['barrio_norm', 'barrio']).map(normalize_barrio)
    out['moneda_norm'] = out.get('moneda', pd.Series(index=out.index, dtype='object')).map(normalize_currency)
    out['id_publicacion'] = out.get('property_id').astype('string')
    out['superficie_ref_m2'] = np.nan
    out['ambientes_ref'] = out.get('ambientes')
    out['banios_ref'] = out.get('banios')
    out['dormitorios_ref'] = out.get('dormitorios')
    out['fecha_extraccion'] = out.get('scraped_at_utc')

    # Airbnb normalizado mensual: el archivo priorizado está pensado para comparar alquiler temporal.
    # En varios registros, precio representa una referencia diaria estimada aunque precio_texto muestre total mensual.
    # Por eso se conserva precio original y se crea una estimación mensual explícita y revisable.
    out['precio_noche_estimado'] = out['precio']
    out['precio_mensual_estimado'] = np.where(out['precio'].notna(), out['precio'] * 30, np.nan)
    out['precio_m2_ref'] = np.nan
    return out

CANONICAL_COLUMNS = [
    'id_publicacion', 'fuente_norm', 'fuente', 'tipo_operacion_norm', 'tipo_operacion',
    'url', 'fecha_extraccion', 'scraped_at_utc', '_archivo_origen',
    'titulo', 'descripcion', 'tipo_propiedad', 'moneda_norm', 'moneda', 'precio',
    'precio_texto', 'precio_m2_ref', 'precio_noche_estimado', 'precio_mensual_estimado',
    'barrio', 'barrio_norm', 'comuna', 'localidad', 'provincia', 'pais',
    'direccion', 'direccion_completa', 'calle', 'altura', 'latitud', 'longitud',
    'superficie_ref_m2', 'superficie_min_m2', 'superficie_max_m2', 'superficie_total_m2', 'superficie_cubierta_m2',
    'ambientes_ref', 'ambientes', 'ambientes_min', 'ambientes_max',
    'dormitorios_ref', 'dormitorios', 'dormitorios_min', 'dormitorios_max',
    'banios_ref', 'banios', 'banios_min', 'banios_max',
    'inmobiliaria', 'seller_id', 'seller_nombre', 'seller_superhost', 'rating', 'reviews',
    'apto_credito', 'balcon', 'balcon_o_patio', 'terraza', 'parrilla', 'pileta',
    'cochera', 'cochera_mencionada', 'amenities_mencionadas', 'precio_sospechoso',
    'parse_ok', 'parse_warnings', 'cantidad_imagenes', 'imagen_principal'
]


def select_canonical(df):
    cols = [c for c in CANONICAL_COLUMNS if c in df.columns]
    return df[cols].copy()


def missingness_correlation(df, max_cols=80):
    flag_cols = [c for c in df.columns if c.endswith('_is_null')]
    num_cols = df.select_dtypes(include='number').columns.tolist()
    cols = list(dict.fromkeys(num_cols + flag_cols))[:max_cols]
    if len(cols) < 2:
        return pd.DataFrame()
    return df[cols].corr(numeric_only=True)

## Revisión de variables categóricas antes de normalizar

Antes de modificar textos conviene mirar labels originales. Esto ayuda a justificar mapeos de barrios, tipo de operación, fuente y moneda.

In [ ]:
cat_vars_relevantes = ['fuente', 'tipo_operacion', 'tipo_propiedad', 'moneda', 'barrio', 'localidad', 'provincia']

for name, df in raw_dfs.items():
    print('\n' + '=' * 100)
    print(name)
    for col in cat_vars_relevantes:
        if col in df.columns:
            print(f'\n-- {col} --')
            display(df[col].astype('string').str.strip().value_counts(dropna=False).head(20).to_frame('conteo'))

## Revisión y parseo de fechas

Se identifican columnas de fecha y se transforman con `pd.to_datetime(errors="coerce", utc=True)`. Las fechas inválidas pasan a nulo y quedan registradas mediante flags.

In [ ]:
for name, df in raw_dfs.items():
    date_like = [c for c in df.columns if any(token in c.lower() for token in ['fecha', 'date', 'scraped_at'])]
    print(name, date_like)
    for col in date_like:
        parsed = pd.to_datetime(df[col], errors='coerce', utc=True)
        print(f'  {col}: válidas={parsed.notna().sum()} | inválidas/nulas={parsed.isna().sum()}')

## Normalización por fuente y operación

Airbnb se trata con cuidado porque ya viene normalizado. No se vuelve a parsear desde cero: se conserva su estructura normalizada, se estandarizan nombres comparables y se agrega una estimación mensual explícita para discutir luego si corresponde usarla en comparaciones.

In [ ]:
df_preprocesado_meli_alq = standardize_meli(df_raw_meli_alq, operacion_forzada='alquiler')
df_preprocesado_meli_alq_temp = standardize_meli(df_raw_meli_alq_temp, operacion_forzada='alquiler_temporal')
df_preprocesado_meli_vtas = standardize_meli(df_raw_meli_vtas, operacion_forzada='venta')
df_preprocesado_argenprop = standardize_portal(df_raw_argenprop, 'argenprop')
df_preprocesado_zonaprop = standardize_portal(df_raw_zonaprop, 'zonaprop')
df_preprocesado_airbnb_temp = standardize_airbnb(df_raw_airbnb_norm)

preprocesados = {
    'df_preprocesado_meli_alq': df_preprocesado_meli_alq,
    'df_preprocesado_meli_alq_temp': df_preprocesado_meli_alq_temp,
    'df_preprocesado_meli_vtas': df_preprocesado_meli_vtas,
    'df_preprocesado_argenprop': df_preprocesado_argenprop,
    'df_preprocesado_zonaprop': df_preprocesado_zonaprop,
    'df_preprocesado_airbnb_temp': df_preprocesado_airbnb_temp,
}

for name, df in preprocesados.items():
    print(name, df.shape)

## Transformación a nulo de valores inválidos

Los valores como `sin datos`, `s/d`, `-99999`, strings vacíos o equivalentes ya fueron transformados a `NaN` por las funciones anteriores. Se verifica el impacto general.

In [ ]:
resumen_nulos_pre = []
for name, df in preprocesados.items():
    resumen_nulos_pre.append({
        'dataframe': name,
        'filas': len(df),
        'columnas': df.shape[1],
        'total_nulos': int(df.isna().sum().sum()),
        'pct_celdas_nulas': round(df.isna().sum().sum() / (df.shape[0] * df.shape[1]) * 100, 2) if df.shape[0] and df.shape[1] else np.nan,
    })

resumen_nulos_pre = pd.DataFrame(resumen_nulos_pre)
display(resumen_nulos_pre)

## Creación de flags de nulidad por fila

Para cada variable con nulos se crea un flag `columna_is_null`. Esto permite analizar si el faltante parece aleatorio o si se asocia con fuente, operación, barrio, precio, superficie u otras variables.

In [ ]:
preprocesados_con_flags = {}
for name, df in preprocesados.items():
    preprocesados_con_flags[name] = add_null_flags(df)
    globals()[name] = preprocesados_con_flags[name]
    print(name, 'flags creados:', sum(c.endswith('_is_null') for c in preprocesados_con_flags[name].columns))

## Matriz de correlación con flags de faltantes

Se calcula correlación entre variables numéricas y flags de nulos. Esto no demuestra causalidad, pero ayuda a detectar mecanismos de ausencia: por ejemplo, si `precio_m2` falta cuando falta superficie, o si ciertos faltantes aparecen más en una fuente que en otra.

In [ ]:
try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_SEABORN = True
except Exception:
    import matplotlib.pyplot as plt
    HAS_SEABORN = False

for name, df in preprocesados_con_flags.items():
    corr = missingness_correlation(df, max_cols=60)
    if corr.empty:
        print(name, 'sin suficientes variables numéricas/flags para correlación')
        continue
    corr.to_csv(REPORTS_DIR / f'correlacion_faltantes_{name}.csv')
    plt.figure(figsize=(14, 10))
    if HAS_SEABORN:
        sns.heatmap(corr, cmap='coolwarm', center=0, vmin=-1, vmax=1)
    else:
        plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
        plt.colorbar()
        plt.xticks(range(len(corr.columns)), corr.columns, rotation=90, fontsize=7)
        plt.yticks(range(len(corr.index)), corr.index, fontsize=7)
    plt.title(f'Matriz de correlación con flags de faltantes - {name}')
    plt.tight_layout()
    plt.show()

## Consolidación con columnas canónicas

Se selecciona un conjunto común de variables para análisis posterior. Las columnas específicas de cada portal se conservan en los datasets preprocesados por fuente; la base consolidada prioriza comparabilidad.

In [ ]:
canonical_frames = []
for name, df in preprocesados_con_flags.items():
    tmp = select_canonical(df)
    tmp['dataset_preprocesado'] = name
    canonical_frames.append(tmp)

df_publicaciones_consolidadas = pd.concat(canonical_frames, ignore_index=True, sort=False)

# Flags también para la base consolidada, pero solo después de definir las columnas comunes.
df_publicaciones_consolidadas = add_null_flags(df_publicaciones_consolidadas)

display(df_publicaciones_consolidadas.head())
print(df_publicaciones_consolidadas.shape)

## Controles rápidos post-consolidación

Estos controles no son EDA final. Sirven para saber si la base integrada quedó usable: volumen por fuente/operación, barrios normalizados, monedas y campos críticos.

In [ ]:
controles = {
    'filas_por_fuente': df_publicaciones_consolidadas['fuente_norm'].value_counts(dropna=False),
    'filas_por_operacion': df_publicaciones_consolidadas['tipo_operacion_norm'].value_counts(dropna=False),
    'monedas': df_publicaciones_consolidadas['moneda_norm'].value_counts(dropna=False),
    'barrios_top_30': df_publicaciones_consolidadas['barrio_norm'].value_counts(dropna=False).head(30),
}

for titulo, serie in controles.items():
    print('\n' + titulo)
    display(serie.to_frame('conteo'))

campos_criticos = ['id_publicacion', 'fuente_norm', 'tipo_operacion_norm', 'precio', 'moneda_norm', 'barrio_norm', 'superficie_ref_m2', 'precio_m2_ref']
display(df_publicaciones_consolidadas[campos_criticos].isna().mean().mul(100).round(2).to_frame('pct_nulos'))

## Indicios rápidos para orientar próximos pasos

Completar luego de ejecutar. Mantener el tono de evidencia preliminar:

- Campos que parecen suficientemente confiables:
- Campos que limitan comparaciones entre fuentes:
- Faltantes que parecen depender de la fuente o tipo de operación:
- Normalizaciones de barrios que conviene revisar manualmente:
- Variables que no deberían usarse todavía en hipótesis:

In [ ]:
# Escribir comentarios breves luego de revisar controles y correlaciones.

## Guardado de datasets procesados

Se guardan bases por fuente/operación y una base consolidada. Estos archivos son los que debería leer el notebook 03.

In [ ]:
outputs = {
    'df_preprocesado_meli_alq': df_preprocesado_meli_alq,
    'df_preprocesado_meli_alq_temp': df_preprocesado_meli_alq_temp,
    'df_preprocesado_meli_vtas': df_preprocesado_meli_vtas,
    'df_preprocesado_argenprop': df_preprocesado_argenprop,
    'df_preprocesado_zonaprop': df_preprocesado_zonaprop,
    'df_preprocesado_airbnb_temp': df_preprocesado_airbnb_temp,
    'df_publicaciones_consolidadas': df_publicaciones_consolidadas,
}

for name, df in outputs.items():
    path = PROCESSED_DIR / f'{name}_v1.csv'
    df.to_csv(path, index=False)
    print('Guardado:', path.relative_to(REPO_ROOT), df.shape)

## Diccionario de datos procesado

Se genera un diccionario base para completar manualmente con definiciones finales, fuente y transformaciones aplicadas.

In [ ]:
diccionario_procesado = []
for col in df_publicaciones_consolidadas.columns:
    diccionario_procesado.append({
        'columna': col,
        'tipo_dato': str(df_publicaciones_consolidadas[col].dtype),
        'significado': 'Completar definición final.',
        'unidad': 'Completar si corresponde.',
        'fuente': 'Base consolidada desde portales inmobiliarios/Airbnb.',
        'transformacion_aplicada': 'Ver notebook 02_limpieza_y_consolidacion.',
        'ejemplo': str(df_publicaciones_consolidadas[col].dropna().iloc[0])[:120] if df_publicaciones_consolidadas[col].notna().any() else np.nan,
    })

diccionario_procesado = pd.DataFrame(diccionario_procesado)
display(diccionario_procesado)
diccionario_procesado.to_csv(PROCESSED_DIR / 'diccionario_datos_procesado_v1.csv', index=False)
print('Guardado:', PROCESSED_DIR / 'diccionario_datos_procesado_v1.csv')

## Celdas pendientes para avanzar

Usar estas celdas si en la revisión aparece una regla adicional de limpieza. No mezclar acá análisis exploratorio final; eso queda para el notebook 03.

In [ ]:
# TODO: agregar mapeos manuales de barrios que queden mal normalizados.

In [ ]:
# TODO: agregar reglas de outliers evidentes de precio/superficie, con justificación.

In [ ]:
# TODO: documentar registros eliminados/corregidos/imputados si se aplica alguna decisión fuerte.